# Comparisons and uncertainty

11 November 2026 · Lecture 2

Run Python cells from the repository root. Prepare local data with `uv run python scripts/fetch_data.py`. Paths are relative to that root.

## About the data

`amazon_purchases.csv` is an extract of Berke and colleagues' Open e-commerce 1.0 dataset. US volunteers donated their Amazon purchase histories. Each row is a product-purchase record with a date, shipping state, category, unit price, quantity and calculated spending in USD. We retain 2018–2022 and seven named product categories. The extract omits participant identifiers and product titles. This convenience sample is not representative of all Amazon customers or total company revenue.

`footwear_weather.csv` matches the Amazon Boot and Sandal records to NOAA's monthly state temperatures, 2018–2022. We summarise these records by calendar month to compare temperature and category shares. `boot_month_coefficients.csv` contains eleven monthly boot-price coefficients relative to January; `boot_summer_contrast.csv` compares the average June–August coefficient with January. The preparation script fits the models and calculates 95% pointwise intervals with clustered standard errors. These are observational associations, not causal effects; the intervals do not correct the voluntary sample's selection bias. No participant identifiers appear in these tables.

Sources, licences, selection rules and limitations are in the [data documentation](../../data/README.md). Files are prepared by `scripts/fetch_data.py`; this notebook reads local data only.

## Questions for today's analysis

Use the plots to investigate these three questions.

1. **What share of boot and sandal purchases cost \$50 or less?**
2. **Do similarly warm months have similar boot-purchase shares?**
3. **Are boots cheaper out of season?**

Each question requires a different comparison: a share below a price threshold,
the relationship between temperature and purchase choice, and uncertainty about
adjusted price differences. We use the same purchase source throughout, with
NOAA temperatures added for question 2.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## Purchase prices: question 1

**What share of boot and sandal purchases cost \$50 or less?**

The hypothetical budget is \$50 per item. Each row is a purchase record, not a
distinct customer. All prices, including the expensive ones, enter the calculation.

How do the mean and median prices compare? What might explain a difference between
them? If the mean exceeds \$50, must most purchases exceed \$50 too?

In [ ]:
purchases = pd.read_csv("data/raw/amazon_purchases.csv", parse_dates=["date"])
# isin() selects either footwear category; copy() gives us a separate working table.
is_footwear = purchases["category"].isin(["Boots", "Sandals"])
footwear = purchases.loc[is_footwear].copy()
print(footwear.head())

In [ ]:
category_groups = footwear.groupby("category")
# print(category_groups.value_counts())
prices_by_category = category_groups["unit_price_usd"]
# The mean is a price; it is not the fraction below a budget.
price_summary = prices_by_category.agg(["count", "mean", "median", "min", "max"])
price_summary.round(2)

### The shape behind the summary: boot prices

Where are most recorded boot prices concentrated? What shape would you expect from
the mean and median in the table?

Each bar below counts purchase records in a \$25 price interval. The two vertical
lines mark the mean and median of all Boot records.

In [ ]:
boot_prices = footwear.loc[footwear["category"].eq("Boots"), "unit_price_usd"]
boot_mean = boot_prices.mean()
boot_median = boot_prices.median()
# Equal-width bin edges start at zero and extend far enough to include the maximum.
price_bins = np.arange(0, boot_prices.max() + 25, 25)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
# hist() counts records in each price interval; no prices are removed.
ax.hist(boot_prices, bins=price_bins, color="#B9CCCA", edgecolor="white")
# Reference lines show where the two summaries fall within the distribution.
ax.axvline(boot_mean, color="#B65A38", linestyle="--", linewidth=2,
           label=f"Mean: ${boot_mean:.2f}")
ax.axvline(boot_median, color="#176D79", linewidth=2,
           label=f"Median: ${boot_median:.2f}")
ax.set(title="Boot prices in the sample", xlabel="Recorded price per item (USD)",
       ylabel="Number of purchase records", xlim=(0, price_bins[-1]))
ax.set_axisbelow(True)
ax.grid(axis="y", alpha=0.2)
ax.legend(frameon=False)
# Attach the sample and bin definition to the exported figure.
fig.text(0.01, 0.02,
         f"Source: Berke et al., 2018–2022. All {len(boot_prices):,} Boot records; bins are $25 wide.\n"
         "Bars count purchase records, not distinct customers. Voluntary sample; prices in nominal USD.",
         fontsize=8)
fig.tight_layout(rect=(0, 0.13, 1, 1))
fig.savefig("figures/02_boot_price_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

### Questions about the histogram

- Which side has the longer tail? How does the shape help explain the positions
  of the mean and median?
- How would a few much more expensive purchases affect each summary?

The histogram shows how records are distributed across price intervals. To answer
the budget question, we need the share of records at or below a chosen price.

### Calculate one point before drawing the curve

Mark purchases costing no more than the budget. The mean of this True/False
column is the fraction within budget: True counts as 1 and False as 0.
Calculate this separately for Boots and Sandals so each has its own denominator.

In [ ]:
budget = 50  # Change this later; no purchase records are removed.
# le() marks prices at or below the budget; the group mean gives the covered share.
footwear["within_budget"] = footwear["unit_price_usd"].le(budget)
print(footwear[["category", "unit_price_usd", "within_budget"]].head())
budget_share = footwear.groupby("category")["within_budget"].mean() * 100

### Cumulative price distributions

A cumulative distribution places the price threshold on the horizontal axis and
the percentage of records at or below that threshold on the vertical axis.
Read up from \$50 to each curve, then across to the percentage. Every price
contributes; there are no histogram bins to choose.

Predict what happens to the percentage when we raise the budget. Could either
curve ever go down? What would a steep section mean?

In [ ]:
category_colours = {"Boots": "#176D79", "Sandals": "#D2643B"}
# fig is the whole figure; ax is its plotting area. figsize is width and height in inches.
fig, ax = plt.subplots(figsize=(9, 5))
# hue draws a curve per category; palette gives it the colour defined above.
# stat="percent" expresses each curve as a percentage of that category's records.
sns.ecdfplot(data=footwear, x="unit_price_usd", hue="category", stat="percent",
             hue_order=["Boots", "Sandals"], palette=category_colours, linewidth=2, ax=ax)

# Mark the budget with a vertical line and highlight its share on each curve.
ax.axvline(budget, color="grey", linestyle="--", linewidth=1)
for category, share in budget_share.items():
    ax.scatter(budget, share, color=category_colours[category], zorder=3)
    # Move labels away from the dots: (horizontal, vertical) offsets in points.
    offset = (0, 12) if category == "Sandals" else (9, -13)
    alignment = "right" if category == "Sandals" else "left"
    # annotate() attaches text to a point; ha chooses which text edge is anchored.
    ax.annotate(f"{category}: {share:.1f}%", (budget, share), xytext=offset,
                textcoords="offset points", ha=alignment, color=category_colours[category])

# Limit the visible price range, keeping all records in the calculation; leave room above 100%.
ax.set(title=f"What share of purchases fits a ${budget:g} budget?",
       xlabel="Price threshold per item (USD)", ylabel="Purchase records at or below the threshold (%)",
       xlim=(0, 350), ylim=(0, 110), yticks=range(0, 101, 20))
ax.grid(axis="y", alpha=0.2)
# Place the note near the figure's bottom left, using coordinates from 0 to 1.
fig.text(0.01, 0.02, "Source: Berke et al., 2018–2022. Boots: 3,081 records; Sandals: 3,577. Each curve uses its own category total.\nAll prices retained. The budget is a demonstration threshold, not a filter. Voluntary purchase histories, not a representative market sample.", fontsize=8)
# Reserve the bottom 13% for the note; rect is (left, bottom, right, top).
fig.tight_layout(rect=(0, 0.13, 1, 1))
# Export at 150 dots per inch; bbox_inches="tight" trims unused outer space.
fig.savefig("figures/02_price_budget_ecdf.png", dpi=150, bbox_inches="tight")
plt.show()

### Questions about the plot

- What percentage of Boot records and Sandal records fall within a \$50 budget?
- Does the share below \$50 agree with what you expected from the mean price?
  How can you explain any difference?
- What would change in the figure if `budget` were 75? Would the curves themselves change?

[Seaborn reference: cumulative distributions](https://seaborn.pydata.org/generated/seaborn.ecdfplot.html).

## Question 2: weather and purchase choice

**Do similarly warm months have similar boot-purchase shares?**

Would you expect the same mix of Boots and
Sandals at a similar temperature in spring and autumn?

The preparation script has already matched shipping state and purchase month
to NOAA's monthly state temperatures. We use 6,502 of the 6,658 footwear records;
129 records have unknown state and 27 use DC, which has no separate statewide series.
The original purchase sample stays unchanged. The state average is an approximate
exposure, not a measurement at the buyer's address.

In [ ]:
weather_purchases = pd.read_csv("data/processed/footwear_weather.csv", parse_dates=["date"])
print("All footwear records:", len(footwear))
print("Records with weather:", len(weather_purchases))
weather_purchases[["date", "state", "category", "temperature_c"]].head()

### Define what one point represents

Use one point per calendar month, pooling 2018–2022. Its horizontal position is
the mean matched temperature across that month's purchase records. Its vertical
position is the percentage of those records that are Boots.

The denominator includes **both Boots and Sandals**. Counting only Boots would
mix up category choice with the total number of recorded purchases. The temperature
mean weights places and years by the number of purchases in this sample; it is
not a national weather average.

In [ ]:
weather_purchases["calendar_month"] = weather_purchases["date"].dt.month
weather_purchases["is_boot"] = weather_purchases["category"].eq("Boots")
# Pool the same calendar month across years; each name defines a summary column.
monthly_weather = weather_purchases.groupby("calendar_month").agg(
    records=("category", "size"),  # size counts all records in the group.
    boots=("is_boot", "sum"),  # Summing True/False counts the Boot records.
    temperature_c=("temperature_c", "mean"),
)
monthly_weather["boot_share"] = monthly_weather["boots"] / monthly_weather["records"] * 100
monthly_weather.head().round(1)

### Temperature and purchase choice

Each point is a monthly summary, not an individual purchase. What relationship do
you see? Are there points with similar temperatures but different boot shares?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.2))
# Each dot is one monthly summary; s sets the marker area in squared points.
ax.scatter(monthly_weather["temperature_c"], monthly_weather["boot_share"],
           s=70, color="#536270")
ax.set(title="Weather and the mix of footwear purchases",
       xlabel="Mean temperature across purchase records (°C)",
       ylabel="Boots as a share of Boot/Sandal records (%)", xlim=(-2, 28), ylim=(0, 100))
ax.grid(alpha=0.2)
fig.text(0.01, 0.02, "Source: Berke et al. + NOAA, 2018–2022. Twelve monthly summaries from 6,502 matched purchase records.\nTemperature is averaged across records; the denominator includes both categories. Descriptive summaries, without adjustment or intervals.\nVoluntary sample and statewide weather: this pattern does not identify a causal temperature effect.", fontsize=8)
# Leave space beneath the axes for the three-line figure note.
fig.tight_layout(rect=(0, 0.17, 1, 1))
plt.show()

### Adding calendar information

The next plot adds month labels and highlights March–May and September–November.
The positions, marker sizes and axes stay the same. What might the month labels
help you understand about the pattern?

In [ ]:
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
# These groups label the same monthly summaries; they do not select new records.
monthly_weather["season"] = "Other months"
monthly_weather.loc[[3, 4, 5], "season"] = "March–May"
monthly_weather.loc[[9, 10, 11], "season"] = "September–November"
season_colours = {"March–May": "#176D79", "September–November": "#B65A38", "Other months": "#A0A5AA"}

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.2))
# Draw each season in its own colour; label supplies its legend entry.
for season, colour in season_colours.items():
    group = monthly_weather.loc[monthly_weather["season"].eq(season)]
    ax.scatter(group["temperature_c"], group["boot_share"], s=70, color=colour, label=season)
# Offset labels in points so they do not cover their markers.
for month, row in monthly_weather.iterrows():
    offset = (7, -13) if month in [2, 3, 4, 5, 6, 7] else (7, 5)
    ax.annotate(month_names[month - 1], (row["temperature_c"], row["boot_share"]),
                xytext=offset, textcoords="offset points", fontsize=9)
# Reuse the previous axis limits so only the colours and labels change.
ax.set(title="Weather and the mix of footwear purchases",
       xlabel="Mean temperature across purchase records (°C)",
       ylabel="Boots as a share of Boot/Sandal records (%)", xlim=(-2, 28), ylim=(0, 100))
ax.grid(alpha=0.2)
ax.legend(title="Calendar months", loc="upper right", frameon=False)
fig.text(0.01, 0.02, "Source: Berke et al. + NOAA, 2018–2022. Twelve monthly summaries from 6,502 matched purchase records.\nTemperature is averaged across records; the denominator includes both categories. Descriptive summaries, without adjustment or intervals.\nVoluntary sample and statewide weather: this pattern does not identify a causal temperature effect.", fontsize=8)
fig.tight_layout(rect=(0, 0.17, 1, 1))
fig.savefig("figures/02_weather_months_reveal.png", dpi=150, bbox_inches="tight")
plt.show()

How do March and November compare? Similar average temperatures can conceal
differences in weather distributions, locations and shoppers.

In [ ]:
# Select March and November by their month-number index for a direct comparison.
monthly_weather.loc[[3, 11], ["records", "temperature_c", "boot_share"]].round(1)

### Questions about the plot

- How similar are March and November in temperature? How similar are their boot shares?
- What do the month labels and colours reveal that was difficult to see in the first plot?
- What might explain the pattern? What evidence would you need to check your explanation?

## Question 3: seasonal boot prices

**Are boots cheaper out of season?**

A seasonal change in the mix of footwear purchases does not itself imply a change
in boot prices. Here, summer means June–August.
We use all 3,081 Boot records, including those without weather matches.

The preparation script fits this OLS model:

`unit_price_usd ~ C(month, Treatment(reference=1)) + C(year) + C(state)`

Each monthly coefficient compares boot prices with January, adjusting for year and
shipping state. January is the reference, so we do not plot it as an estimated zero
with a zero-width interval. Prices are nominal USD; changes in products and purchasers
can affect the comparisons. This is a seasonal coefficient plot, not a causal event study.

In [ ]:
# Load the prepared OLS estimates and interval bounds; no model is refitted here.
price_effects = pd.read_csv("data/processed/boot_month_coefficients.csv")
print(price_effects.round(2))

### Monthly price differences

Are the summer estimates positive or negative? Does the line alone tell you which
differences you can distinguish from zero?

In [ ]:
# Set limits using the intervals now, so adding uncertainty does not change the scale.
interval_range = price_effects["ci_high"].max() - price_effects["ci_low"].min()
price_ylim = (price_effects["ci_low"].min() - 0.05 * interval_range,
              price_effects["ci_high"].max() + 0.05 * interval_range)
month_labels = ["Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
fig, ax = plt.subplots(figsize=(9, 4.5))
# "o-" draws circular markers connected by a line to guide the eye.
ax.plot(price_effects["month"], price_effects["estimate"], "o-")
# Zero means no adjusted price difference relative to January.
ax.axhline(0, color="grey", linewidth=1)
ax.set(title="Boot prices compared with January", xlabel="Purchase month (January is the reference)",
       ylabel="Adjusted price difference (USD)", xticks=price_effects["month"],
       xticklabels=month_labels, ylim=price_ylim)
fig.text(0.01, 0.02, "Berke et al., 2018–2022. OLS with month, year and state indicators; 3,081 boot records.\nCoefficients only; uncertainty not yet shown. Connected points guide the eye; voluntary sample.", fontsize=8)
fig.tight_layout(rect=(0, 0.13, 1, 1))
plt.show()

### Confidence intervals for the price differences

The next plot adds 95% pointwise confidence intervals, using standard errors clustered
by participant (1,392 participants). The coefficients, line and axis limits stay the same.
These describe uncertainty about price differences, not the distribution of individual prices.

In [ ]:
# errorbar needs distances from each estimate, not the interval endpoints themselves.
lower_distance = price_effects["estimate"] - price_effects["ci_low"]
upper_distance = price_effects["ci_high"] - price_effects["estimate"]
# The first row gives downward distances; the second gives upward distances.
errors = [lower_distance, upper_distance]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
# yerr adds the intervals; capsize sets the short horizontal marks at their ends.
ax.errorbar(price_effects["month"], price_effects["estimate"], yerr=errors, fmt="o-", capsize=4)
ax.axhline(0, color="grey", linewidth=1)
# Keep the same month labels and vertical scale as the coefficients-only plot.
ax.set(title="Boot prices compared with January", xlabel="Purchase month (January is the reference)",
       ylabel="Adjusted price difference (USD)", xticks=price_effects["month"],
       xticklabels=month_labels, ylim=price_ylim)
fig.text(0.01, 0.02, "Berke et al., 2018–2022. OLS with month, year and state indicators; 3,081 boot records.\n95% pointwise t intervals; SEs clustered by participant (1,392 clusters). Voluntary sample.", fontsize=8)
fig.tight_layout(rect=(0, 0.13, 1, 1))
fig.savefig("figures/02_boot_month_coefficients.png", dpi=150, bbox_inches="tight")
plt.show()

### Questions about the intervals

- Which summer intervals include zero? What can you conclude about those months' prices
  relative to January?
- How does adding the intervals change your interpretation of the line?

The intervals are separate 95% intervals, not a 95% band for all eleven comparisons.
Nor do they test differences between pairs of non-reference months.

The average of the June, July and August coefficients summarises the summer comparison.
Its interval uses their covariance rather than averaging the three interval endpoints.
The preparation script supplies this comparison alongside the monthly estimates.

In [ ]:
# This file contains the joint summer comparison, including its model-based interval.
summer_comparison = pd.read_csv("data/processed/boot_summer_contrast.csv")
summer_comparison.round(2)

### Returning to the summer-price question

- Do the summer estimate and its interval support the claim that boots are cheaper
  in summer than in January? Which values support your conclusion?
- What is the difference between an interval that includes zero and evidence that
  prices are identical?

Model formulas, sample restrictions, sources and interval construction are in the
[data documentation](../../data/README.md#footwear-regressions-and-monthly-weather).

In [the exercise](exercises.ipynb), redesign a chart to compare seasonal purchasing
patterns for Boots and Sandals.

Course AI use and disclosure requirements are in [the policy](../../docs/ai_policy.md). If you use assistance, verify the code, numbers and interpretation and disclose it as required.